In [30]:
using Pkg
Pkg.activate("/Users/jiyong/.juliaenv/numerical")
using ForwardDiff


  Activating project at `~/.juliaenv/numerical`


In [2]:
g(x::Vector) = x[1]+ 2*x[2]
ForwardDiff.gradient(g, [1.0, 1.0])

2-element Vector{Float64}:
 1.0
 2.0

In [3]:
f(x::Real) = x*sin(x)
ForwardDiff.derivative(f, π)

-3.141592653589793

In [5]:
H(x::Vector) = [x[1]*x[2], x[1]+x[2]]
ForwardDiff.jacobian(H, [1.0, 2.0])

2×2 Matrix{Float64}:
 2.0  1.0
 1.0  1.0

In [24]:

struct DNumber{T<:Real}
    v::T
    e::T

    function DNumber(v::Real, e::Union{Real, Nothing}=nothing) 
        if e == nothing
            e=oneunit(v)
            new{typeof(v)}(v, e)
        else
            S = promote_type(typeof(v), typeof(e))
            new{S}(v, e)
        end
    end

    function DNumber{T}(v::Real, e::Union{Real, Nothing}=nothing) where T<:Union{Float16, Float32, Float64}
        if e == nothing
            new{T}(convert(T, v), one(T))
        else
            new{S}(convert(T, v), e)
        end
    end
    end

Base.:+(x::DNumber, y::DNumber) = DNumber(x.v + y.v, x.e + y.e)
Base.:-(x::DNumber, y::DNumber) = DNumber(x.v - y.v, x.e - y.e)
Base.:*(x::DNumber, y::DNumber) = DNumber(x.v * y.v, x.e * y.v + x.v * y.e)
Base.:/(x::DNumber, y::DNumber) = DNumber(x.v / y.v, (x.e * y.v - x.v * y.e) / y.v^2)
Base.:-(x::DNumber) = DNumber(-x.v, -x.e)
Base.:+(x::DNumber, y::Real) = DNumber(x.v + y, x.e)
Base.:+(x::Real, y::DNumber) = y+x
Base.:-(x::DNumber, y::Real) = DNumber(x.v-y, x.e)
Base.:-(x::Real, y::DNumber) = -(y-x)
Base.:*(x::DNumber, y::Number) = x*DNumber(y, zero(y))
Base.:*(x::Number, y::DNumber) = DNumber(x, zero(x))*y
Base.:/(x::DNumber, y::Number) = x/DNumber(y, zero(y))
Base.:/(x::Number, y::DNumber) = DNumber(x, zero(x))/y
Base.:^(x::DNumber, y::Real) = DNumber((x.v)^y, y*((x.v)^(y-1))*x.e)

import Base.sin, Base.cos, Base.tan, Base.exp, Base.log
sin(x::DNumber) = DNumber(sin(x.v), cos(x.v)*x.e)
cos(x::DNumber) = DNumber(cos(x.v), -sin(x.v)*x.e)  
tan(x::DNumber) = sin(x)/cos(x)
exp(x::DNumber) = DNumber(exp(x.v), exp(x.v)*x.e)
log(x::DNumber) = DNumber(log(x.v), x.e/x.v)


function deriv(f::Function, x::Real)
    f(DNumber(x)).e
end

function deriv(f::Function, x::DNumber)
    f(x).e
end



deriv (generic function with 2 methods)

In [25]:

DNumber{Float64}(3.0f0)

DNumber{Float64}(3.0, 1.0)

In [27]:
DNumber(2.0, 1.0)^5

DNumber{Float64}(32.0, 80.0)

In [28]:
f(x) = x^2
g(x) = 2*x
deriv(g∘f, DNumber(1.0, 0.3))

1.2

In [14]:
3*cos(π/3)*(sin(π/3))^2

1.1250000000000002

In [36]:
cs=[]
ForwardDiff.derivative!(cs, x->x^2, 1.0)

MethodError: MethodError: no method matching extract_derivative!(::Type{ForwardDiff.Tag{var"#31#32", Float64}}, ::Vector{Any}, ::ForwardDiff.Dual{ForwardDiff.Tag{var"#31#32", Float64}, Float64, 1})
The function `extract_derivative!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  extract_derivative!(::Type{T}, !Matched::DiffResults.DiffResult, ::Any) where T
   @ ForwardDiff ~/.julia/packages/ForwardDiff/UBbGT/src/derivative.jl:98
  extract_derivative!(::Type{T}, ::AbstractArray, !Matched::AbstractArray) where T
   @ ForwardDiff ~/.julia/packages/ForwardDiff/UBbGT/src/derivative.jl:96


In [33]:
typeof(qs)

Float64